In [2]:
%pip install "transformers==4.44.2" "trl==0.11.4" "peft==0.13.2" "accelerate==0.34.2" "datasets==2.21.0" "numpy<2"

/home/ankitanand/Documents/pp/Finetuning_HF/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch, transformers, trl, peft
print(torch.__version__, torch.cuda.is_available())
print(transformers.__version__, trl.__version__, peft.__version__)

from trl import setup_chat_format, DataCollatorForCompletionOnlyLM
from trl.extras.dataset_formatting import FORMAT_MAPPING, instructions_formatting_function, conversations_formatting_function
from trl.trainer import ConstantLengthDataset
print("✅ book imports OK")

/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.13.0+cu130 True
4.44.2 0.11.4 0.13.2
✅ book imports OK


In [4]:
import torch
from datasets import load_dataset, Dataset
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, DataCollatorForLanguageModeling, DataCollatorWithPadding, DataCollatorWithFlattening, BitsAndBytesConfig
from trl.extras.dataset_formatting import FORMAT_MAPPING, instructions_formatting_function, conversations_formatting_function
from trl.trainer import ConstantLengthDataset

In [5]:
tokenizer = AutoTokenizer.from_pretrained('facebook/opt-350m')
quote = 'A noble spirit embiggens the smallest man.'
print(tokenizer.tokenize(quote))
print(tokenizer.encode(quote, add_special_tokens=False))


['A', 'Ġnoble', 'Ġspirit', 'Ġemb', 'igg', 'ens', 'Ġthe', 'Ġsmallest', 'Ġman', '.']
[250, 25097, 4780, 18484, 11702, 1290, 5, 15654, 313, 4]


/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
repo_id = "microsoft/phi-3-mini-4k-instruct"
tokenizer_phi = AutoTokenizer.from_pretrained(repo_id)
print(tokenizer_phi.chat_template)

{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}


In [10]:
messages = [
    {"role": "system",    "content": "You are a helpful AI assistant."},
    {"role": "user",      "content": "What is the capital of Argentina?"},
    {"role": "assistant", "content": "Buenos Aires."},
]

formatted = tokenizer_phi.apply_chat_template(
  conversation=messages, tokenize=False, add_generation_prompt=False
)
print(formatted)

<|system|>
You are a helpful AI assistant.<|end|>
<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>
Buenos Aires.<|end|>
<|endoftext|>


In [11]:
inference_input = tokenizer_phi.apply_chat_template(
    conversation=messages[:-1], tokenize=False, add_generation_prompt=True
)
print(inference_input)

<|system|>
You are a helpful AI assistant.<|end|>
<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>



In [12]:
{"messages": [
  {"role": "system", "content": "<directives>"},
  {"role": "user", "content": "<prompt>"},
  {"role": "assistant", "content": "<completion>"}
]}

{'messages': [{'role': 'system', 'content': '<directives>'},
  {'role': 'user', 'content': '<prompt>'},
  {'role': 'assistant', 'content': '<completion>'}]}

In [13]:
{"prompt": "<prompt>", "completion": "<completion>"}

{'prompt': '<prompt>', 'completion': '<completion>'}

In [15]:
conversation_ds = Dataset.from_list([{'messages': messages}])   
conversation_ds.features

{'messages': [{'role': Value(dtype='string', id=None),
   'content': Value(dtype='string', id=None)}]}

In [16]:
FORMAT_MAPPING['chatml'] == conversation_ds.features['messages']

True

In [17]:
formatting_func = conversations_formatting_function(
    tokenizer_phi, messages_field='messages'
)
print(formatting_func(conversation_ds[0]))  

<|system|>
You are a helpful AI assistant.<|end|>
<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>
Buenos Aires.<|end|>
<|endoftext|>


In [19]:
def format_dataset(examples):
    if isinstance(examples[messages_field][0], list):
        output_texts = []
        for i in range(len(examples[messages_field])):
            output_texts.append(
                tokenizer.apply_chat_template(
                    examples[messages_field][i], tokenize=False
                )
            )
        return output_texts
    else:
        return tokenizer.apply_chat_template(examples[messages_field], tokenize=False)

In [21]:
instructions = [{'prompt': 'What is the capital of Argentina?',
                'completion': 'Buenos Aires.'}]
instruction_ds = Dataset.from_list(instructions)
instruction_ds.features

{'prompt': Value(dtype='string', id=None),
 'completion': Value(dtype='string', id=None)}

In [22]:
FORMAT_MAPPING['instruction'] == instruction_ds.features

True

In [23]:
formatting_func = instructions_formatting_function(tokenizer_phi)
formatting_func

<function trl.extras.dataset_formatting.instructions_formatting_function.<locals>.format_dataset(examples)>

In [24]:
# formatting function for instruction format
def format_dataset(examples):
    if isinstance(examples["prompt"], list):
        output_texts = []
        for i in range(len(examples["prompt"])):
            converted_sample = [
                {"role": "user", "content": examples["prompt"][i]},
                {"role": "assistant", "content": examples["completion"][i]},
            ]
            output_texts.append(
                tokenizer.apply_chat_template(converted_sample, tokenize=False)
            )
        return output_texts
    else:
        converted_sample = [
            {"role": "user", "content": examples["prompt"]},
            {"role": "assistant", "content": examples["completion"]},
        ]
        return tokenizer.apply_chat_template(converted_sample, tokenize=False)

In [25]:
def byo_formatting_func1(examples):
    messages = examples["messages"] 
    output_texts = tokenizer_phi.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return output_texts

In [26]:
ds_msg = Dataset.from_dict({'messages': batch_messages})    
ds_msg.map(lambda v: tokenizer_phi(byo_formatting_func1(v)), batched=True)  

NameError: name 'batch_messages' is not defined

In [27]:
# book p.119 — the batch of conversations
batch_messages = [
    [{'role': 'user',      'content': 'What is the capital of Argentina?'},
     {'role': 'assistant', 'content': 'Buenos Aires.'}],
    [{'role': 'user',      'content': 'What is the capital of the United States?'},
     {'role': 'assistant', 'content': 'Washington D.C.'}]
]

# book p.119 — BYOFF: Bring Your Own Formatting Function
def byo_formatting_func1(examples):
    messages = examples["messages"]
    output_texts = tokenizer_phi.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return output_texts

In [28]:
ds_msg = Dataset.from_dict({'messages': batch_messages})
ds_msg.map(lambda v: tokenizer_phi(byo_formatting_func1(v)), batched=True)

Map: 100%|██████████| 2/2 [00:00<00:00, 99.78 examples/s]


Dataset({
    features: ['messages', 'input_ids', 'attention_mask'],
    num_rows: 2
})

In [29]:
def byo_formatting_func2(examples):
    instruction_template = '### Question:'
    response_template = '### Answer:'
    text = f"{instruction_template} {examples['prompt']}\n"
    text += f"{response_template} {examples['completion']}"
    text += tokenizer_phi.eos_token
    return text

In [31]:
# ---- p.119: the prompt/completion batch ----
batch_prompts_completions = {
    'prompt':     ['What is the capital of Argentina?',
                   'What is the capital of the United States?'],
    'completion': ['Buenos Aires.',
                   'Washington D.C.']
}

# ---- p.120: BYOFF #2 — custom template, single example only ----
def byo_formatting_func2(examples):
    instruction_template = '### Question:'
    response_template = '### Answer:'
    text  = f"{instruction_template} {examples['prompt']}\n"
    text += f"{response_template} {examples['completion']}"
    text += tokenizer_phi.eos_token
    return text

# ---- p.121: BYOFF #3 — same thing, but batch-safe (adds the loop) ----
def byo_formatting_func3(examples):
    output_texts = []
    instruction_template = '### Question:'
    response_template = '### Answer:'
    for i in range(len(examples['prompt'])):
        text  = f"{instruction_template} {examples['prompt'][i]}\n"
        text += f"{response_template} {examples['completion'][i]}"
        text += tokenizer_phi.eos_token
        output_texts.append(text)
    return output_texts

In [32]:
ds_prompt = Dataset.from_dict(batch_prompts_completions)
print(byo_formatting_func2(ds_prompt[0]))

### Question: What is the capital of Argentina?
### Answer: Buenos Aires.<|endoftext|>


In [35]:
ds_prompt.map(lambda v: tokenizer_phi(byo_formatting_func2(v)))

Map: 100%|██████████| 2/2 [00:00<00:00, 1024.63 examples/s]


Dataset({
    features: ['prompt', 'completion', 'input_ids', 'attention_mask'],
    num_rows: 2
})

In [37]:
def byo_formatting_func3(examples):
    output_texts = []
    instruction_template = '### Question:'
    response_template = '### Answer:'
    for i in range(len(examples['prompt'])):
        text = f"{instruction_template} {examples['prompt'][i]}\n"
        text += f"{response_template} {examples['completion'][i]}"
        text += tokenizer_phi.eos_token
        output_texts.append(text)
    return output_texts

ds_prompt.map(lambda v: tokenizer_phi(byo_formatting_func3(v)), batched=True)

Map: 100%|██████████| 2/2 [00:00<00:00, 969.56 examples/s]


Dataset({
    features: ['prompt', 'completion', 'input_ids', 'attention_mask'],
    num_rows: 2
})

In [38]:
def byofd_formatting_func(examples):
    messages = examples["messages"]
    output_texts = tokenizer_phi.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {'text': output_texts}

In [39]:
formatted_ds = ds_msg.map(byofd_formatting_func, batched=True)
formatted_ds['text']

Map: 100%|██████████| 2/2 [00:00<00:00, 808.15 examples/s]


['<|user|>\nWhat is the capital of Argentina?<|end|>\n<|assistant|>\nBuenos Aires.<|end|>\n<|endoftext|>',
 '<|user|>\nWhat is the capital of the United States?<|end|>\n<|assistant|>\nWashington D.C.<|end|>\n<|endoftext|>']